In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d anishsarkar22/asvpoof-2019-dataset-la
!unzip asvpoof-2019-dataset-la.zip

In [ ]:
import os
import numpy as np
import librosa
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("All libraries imported successfully")

In [ ]:
train_protocol = pd.read_csv(
    'LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt',
    sep=' ', header=None,
    names=['speaker', 'file', 'env', 'attack', 'label']
)

print(train_protocol.head())
print(train_protocol['label'].value_counts())

In [ ]:
def extract_mfcc(file_path, n_mfcc=40, max_len=200):
    try:
        audio, sr = librosa.load(file_path, sr=16000)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
        if mfcc.shape[1] < max_len:
            mfcc = np.pad(mfcc, ((0,0),(0, max_len - mfcc.shape[1])))
        else:
            mfcc = mfcc[:, :max_len]
        return mfcc.T
    except:
        return None

# Use subset for speed (full dataset = very slow on Colab)
subset = train_protocol.sample(n=2000, random_state=42)
print(subset['label'].value_counts())

In [ ]:
X, y = [], []
train_audio_path = 'LA/ASVspoof2019_LA_train/flac/'

for _, row in subset.iterrows():
    path = os.path.join(train_audio_path, row['file'] + '.flac')
    mfcc = extract_mfcc(path)
    if mfcc is not None:
        X.append(mfcc)
        y.append(row['label'])

X = np.array(X)
y = np.array(y)
print(f"X shape: {X.shape}, y shape: {y.shape}")

In [ ]:
# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {le.classes_}")

In [ ]:
model = Sequential([
    LSTM(128, input_shape=(200, 40), return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Model Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Model Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

In [ ]:
# Evaluation
y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()

print(classification_report(y_test, y_pred, target_names=le.classes_))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

# Rebuild model
model2 = Sequential([
    tf.keras.layers.Input(shape=(200, 40)),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history2 = model2.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_data=(X_test, y_test),
    class_weight=class_weight_dict,
    verbose=1
)

In [ ]:
bonafide_df = train_protocol[train_protocol['label'] == 'bonafide']
spoof_df = train_protocol[train_protocol['label'] == 'spoof'].sample(n=len(bonafide_df), random_state=42)
balanced_df = pd.concat([bonafide_df, spoof_df]).sample(frac=1, random_state=42)

print(balanced_df['label'].value_counts())

In [ ]:
X2, y2 = [], []

for _, row in balanced_df.iterrows():
    path = os.path.join(train_audio_path, row['file'] + '.flac')
    mfcc = extract_mfcc(path)
    if mfcc is not None:
        X2.append(mfcc)
        y2.append(row['label'])

X2 = np.array(X2)
y2 = np.array(y2)
print(f"X2 shape: {X2.shape}, y2 shape: {y2.shape}")

In [ ]:
le2 = LabelEncoder()
y2_encoded = le2.fit_transform(y2)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2_encoded, test_size=0.2, random_state=42, stratify=y2_encoded
)

model3 = Sequential([
    tf.keras.layers.Input(shape=(200, 40)),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history3 = model3.fit(
    X2_train, y2_train,
    epochs=15,
    batch_size=32,
    validation_data=(X2_test, y2_test),
    verbose=1
)

In [ ]:
y2_pred = (model3.predict(X2_test) > 0.5).astype(int).flatten()

print(classification_report(y2_test, y2_pred, target_names=le2.classes_))

cm2 = confusion_matrix(y2_test, y2_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues',
            xticklabels=le2.classes_, yticklabels=le2.classes_)
plt.title('Confusion Matrix - Balanced Model')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('confusion_matrix_balanced.png', dpi=150)
plt.show()

In [ ]:
# Save model !!
model3.save('audio_deepfake_detector.h5')
print("Model saved.")

# Final training plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history3.history['accuracy'], label='Train')
axes[0].plot(history3.history['val_accuracy'], label='Validation')
axes[0].set_title('Model Accuracy (Balanced)')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history3.history['loss'], label='Train')
axes[1].plot(history3.history['val_loss'], label='Validation')
axes[1].set_title('Model Loss (Balanced)')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_history_balanced.png', dpi=150)
plt.show()